# LeetCode 2077 — Paths in Maze That Lead to Same Room bằng Python

Link đề: https://leetcode.com/problems/paths-in-maze-that-lead-to-same-room/ · Độ khó: **Medium** · Chủ đề: Graph

> **Bài Premium.** Đề trong notebook được viết lại theo hiểu biết về bài (nguồn: doocs/leetcode). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def numberOfPaths(self, n: int, corridors: List[List[int]]) -> int:
        ...
```

Bài toán thực chất là **đếm tam giác trong đồ thị vô hướng**. Lộ trình:
1. **Brute force bộ ba** `a < b < c` với ma trận kề — `O(n³)`.
2. **Mỗi cạnh `(u, v)` đếm láng giềng chung** `|N(u) ∩ N(v)|`, chia 3.
3. **Đếm có thứ tự** `u < v < w`: không cần chia, mỗi tam giác được đếm đúng một lần.
4. **Định hướng cạnh theo bậc** (thấp → cao): `O(E·√E)` — kỹ thuật kinh điển.
5. **Bitset** bằng số nguyên Python: `popcount(adj[u] & adj[v])`.

## 1. Mục tiêu học

1. Dịch đề sang ngôn ngữ đồ thị: "chu trình độ dài 3, khác nhau theo **tập phòng**" = **tam giác** (bộ 3 đỉnh đôi một kề nhau).
2. Hiểu vì sao đếm theo cạnh bị lặp **3 lần** và đếm theo đỉnh không thứ tự bị lặp **6 lần** — và cách khử lặp bằng ràng buộc thứ tự.
3. Dùng **tập láng giềng** (`set`) và phép giao `&` để đếm láng giềng chung.
4. Biết kỹ thuật **định hướng theo bậc** cho chặn trên `O(E√E)` và vì sao mỗi đỉnh có tối đa `O(√E)` cạnh đi ra.
5. Biết dùng **bitset** (`int` Python + `bit_count`) để tăng tốc hằng số.

## 2. Đề bài & ví dụ

Mê cung có `n` phòng đánh số `1..n`; `corridors[i] = [a, b]` là hành lang **hai chiều** nối phòng `a` và `b`. **Điểm rối** của mê cung là số **chu trình độ dài 3** khác nhau, ví dụ `1 → 2 → 3 → 1`. Hai chu trình khác nhau nếu có ít nhất một phòng thuộc chu trình này mà không thuộc chu trình kia — tức là chu trình được xác định bởi **tập 3 phòng**.

- `1 → 2 → 3 → 4` không phải chu trình (không quay về).
- `1 → 2 → 3 → 2 → 1` có độ dài 4 và lặp phòng — không tính.

**Ví dụ 1:** `n = 5`, `corridors = [[1,2],[5,2],[4,1],[2,4],[3,1],[3,4]]` → `2` (tam giác `{1,3,4}` và `{1,2,4}`).

**Ví dụ 2:** `n = 4`, `corridors = [[1,2],[3,4]]` → `0`.

**Ràng buộc:** `2 ≤ n ≤ 1000`, `1 ≤ len(corridors) ≤ 5·10⁴`, `a ≠ b`, không có hành lang trùng.

In [1]:
import random
import time
from itertools import combinations
from typing import List

examples = [((5, [[1, 2], [5, 2], [4, 1], [2, 4], [3, 1], [3, 4]]), 2),
            ((4, [[1, 2], [3, 4]]), 0)]
for (n, cor), want in examples:
    print(f"n = {n}, corridors = {cor} -> kỳ vọng {want}")

n = 5, corridors = [[1, 2], [5, 2], [4, 1], [2, 4], [3, 1], [3, 4]] -> kỳ vọng 2
n = 4, corridors = [[1, 2], [3, 4]] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Danh sách kề bằng `set`

Với đồ thị vô hướng, mỗi cạnh `(a, b)` phải được thêm **hai chiều**. Dùng `set` để kiểm tra kề trong `O(1)` và giao hai tập nhanh.

In [2]:
def build_adj(n, corridors):
    adj = [set() for _ in range(n + 1)]       # phòng đánh số từ 1
    for a, b in corridors:
        adj[a].add(b)
        adj[b].add(a)
    return adj

adj = build_adj(*examples[0][0])
for v in range(1, 6):
    print(v, "->", sorted(adj[v]))

1 -> [2, 3, 4]
2 -> [1, 4, 5]
3 -> [1, 4]
4 -> [1, 2, 3]
5 -> [2]


### 3.2 Giao hai tập `&` và độ phức tạp

`A & B` tốn `O(min(|A|, |B|))` trong CPython (duyệt tập nhỏ, tra trong tập lớn). `len(A & B)` cho số láng giềng chung.

In [3]:
print("N(1) & N(4) =", adj[1] & adj[4], "-> 2 láng giềng chung => cạnh (1,4) nằm trong 2 tam giác")
print("N(2) & N(5) =", adj[2] & adj[5])

N(1) & N(4) = {2, 3} -> 2 láng giềng chung => cạnh (1,4) nằm trong 2 tam giác
N(2) & N(5) = set()


### 3.3 `combinations` để liệt kê bộ ba không thứ tự

In [4]:
print(list(combinations([1, 2, 3, 4], 3)))
print("C(1000, 3) =", 1000 * 999 * 998 // 6, "bộ ba — quá nhiều để duyệt hết trong Python")

[(1, 2, 3), (1, 2, 4), (1, 3, 4), (2, 3, 4)]
C(1000, 3) = 166167000 bộ ba — quá nhiều để duyệt hết trong Python


### 3.4 Bitset bằng `int` Python

Biểu diễn `N(v)` bằng số nguyên có bit `u` bật nếu `u` kề `v`. `(x & y).bit_count()` (Python ≥ 3.10) đếm số láng giềng chung — mọi thứ chạy trong C trên các từ máy 64 bit.

In [5]:
bits = [0] * 6
for a, b in examples[0][0][1]:
    bits[a] |= 1 << b
    bits[b] |= 1 << a
print("bits[1] =", bin(bits[1]), "| bits[4] =", bin(bits[4]), "| chung:", (bits[1] & bits[4]).bit_count())

bits[1] = 0b11100 | bits[4] = 0b1110 | chung: 2


## 4. Ý tưởng nền tảng / trực giác

**Tam giác** `{a, b, c}`: ba cạnh `ab`, `bc`, `ca` đều tồn tại. Một tam giác ứng với **6 chu trình có hướng có điểm xuất phát** (`3` điểm bắt đầu × `2` chiều) nhưng đề chỉ tính **1** (theo tập phòng).

Tính tay ví dụ 1 — với từng cạnh, đếm láng giềng chung:

| cạnh | N(u) | N(v) | chung | tam giác chứa cạnh |
|---|---|---|---|---|
| (1,2) | {2,3,4} | {1,4,5} | {4} | {1,2,4} |
| (5,2) | {2} | {1,4,5} | ∅ | — |
| (4,1) | {1,2,3} | {2,3,4} | {2,3} | {1,2,4}, {1,3,4} |
| (2,4) | {1,4,5} | {1,2,3} | {1} | {1,2,4} |
| (3,1) | {1,4} | {2,3,4} | {4} | {1,3,4} |
| (3,4) | {1,4} | {1,2,3} | {1} | {1,3,4} |

Tổng cột "chung" = `1 + 0 + 2 + 1 + 1 + 1 = 6`. Mỗi tam giác có **3 cạnh** nên được đếm 3 lần ⇒ `6 / 3 = 2`. ✓

In [6]:
total = 0
for a, b in examples[0][0][1]:
    common = adj[a] & adj[b]
    total += len(common)
    print(f"cạnh ({a},{b}): chung = {sorted(common)}")
print("tổng =", total, "-> chia 3 =", total // 3)

cạnh (1,2): chung = [4]
cạnh (5,2): chung = []
cạnh (4,1): chung = [2, 3]
cạnh (2,4): chung = [1]
cạnh (3,1): chung = [4]
cạnh (3,4): chung = [1]
tổng = 6 -> chia 3 = 2


## 5. Cách 1 — Brute Force / baseline: duyệt mọi bộ ba

**Ý tưởng:** với mọi `a < b < c`, kiểm tra ba cạnh bằng ma trận kề (`O(1)` mỗi lần).

In [7]:
def triangles_brute(n, corridors):
    M = [[False] * (n + 1) for _ in range(n + 1)]
    for a, b in corridors:
        M[a][b] = M[b][a] = True
    cnt = 0
    for a in range(1, n + 1):
        for b in range(a + 1, n + 1):
            if not M[a][b]:
                continue                         # cắt tỉa: không có cạnh ab thì bỏ cả nhánh c
            for c in range(b + 1, n + 1):
                if M[a][c] and M[b][c]:
                    cnt += 1
    return cnt

for (n, cor), want in examples:
    print(triangles_brute(n, cor), "kỳ vọng", want)

2 kỳ vọng 2
0 kỳ vọng 0


**Độ phức tạp:** không cắt tỉa là `O(n³)` ≈ `1.7·10⁸` bộ ba với `n = 1000` — quá chậm trong Python. Có cắt tỉa thì thành `O(E · n)` (mỗi cạnh `ab` với `a < b` duyệt `c`) ≈ `5·10⁷` — vẫn chậm. Space `O(n²)` cho ma trận.

**Nút thắt:** duyệt `c` qua **mọi** đỉnh, trong khi chỉ những `c` kề cả `a` lẫn `b` mới có ích — chính là **giao hai tập láng giềng**.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** số tam giác chứa cạnh `(u, v)` = `|N(u) ∩ N(v)|`. Tổng trên mọi cạnh đếm mỗi tam giác đúng **3** lần.

**Insight 2 (khử lặp bằng thứ tự):** chỉ đếm tam giác tại cạnh "nhỏ nhất" của nó. Cụ thể: với `u < v`, chỉ đếm `w ∈ N(u) ∩ N(v)` có `w > v`. Mỗi tam giác `{a < b < c}` được đếm đúng một lần (tại cạnh `(a, b)` với `w = c`).

**Insight 3 (định hướng theo bậc):** sắp xếp đỉnh theo `(bậc, chỉ số)`; định hướng mỗi cạnh từ đỉnh "thấp" sang đỉnh "cao". Gọi `out(v)` = các đỉnh cao hơn kề `v`. Mỗi tam giác có đúng một đỉnh thấp nhất `x`, đỉnh giữa `y`, đỉnh cao nhất `z`; nó được đếm đúng một lần khi xét cạnh `x → y` và thấy `z ∈ out(x) ∩ out(y)`.

Quy tắc: *nếu* `|out(v)|` đều nhỏ (`≤ √(2E)`) *thì* tổng chi phí giao là `O(E √E)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Giao láng giềng theo từng cạnh, chia 3

In [8]:
def triangles_edge_div3(n, corridors):
    adj = build_adj(n, corridors)
    total = 0
    for a, b in corridors:
        total += len(adj[a] & adj[b])
    return total // 3

for (n, cor), want in examples:
    print(triangles_edge_div3(n, cor), "kỳ vọng", want)

2 kỳ vọng 2
0 kỳ vọng 0


**Độ phức tạp:** `O(Σ_{(u,v)} min(deg u, deg v))` — chặn trên `O(E · √E)` (bổ đề ở mục 10). Space `O(n + E)`.

### 7.2 Cách 3 — Đếm có thứ tự `u < v < w` (không cần chia)

In [9]:
def triangles_ordered(n, corridors):
    adj = build_adj(n, corridors)
    cnt = 0
    for u in range(1, n + 1):
        for v in adj[u]:
            if v > u:
                for w in adj[u] & adj[v]:
                    if w > v:
                        cnt += 1
    return cnt

for (n, cor), want in examples:
    print(triangles_ordered(n, cor), "kỳ vọng", want)

2 kỳ vọng 2
0 kỳ vọng 0


Cách này dễ chứng minh (mục 10) và tránh nhầm hệ số chia. Chi phí giống Cách 2 nhưng có thêm vòng `for w` trong Python — chậm hơn một chút.

Biến thể gọn hơn: lưu `higher[u] = {v ∈ N(u) : v > u}`; khi đó số tam giác = `Σ_{u} Σ_{v ∈ higher[u]} |higher[u] ∩ higher[v]|`.

### 7.3 Cách 4 — Định hướng cạnh theo bậc (code nộp)

Thứ hạng `rank(v) = (deg v, v)`. Cạnh `(a, b)` hướng từ đỉnh có `rank` nhỏ hơn sang đỉnh có `rank` lớn hơn.

In [10]:
def triangles_degree_oriented(n, corridors):
    deg = [0] * (n + 1)
    for a, b in corridors:
        deg[a] += 1; deg[b] += 1
    out = [set() for _ in range(n + 1)]
    for a, b in corridors:
        if (deg[a], a) < (deg[b], b):
            out[a].add(b)
        else:
            out[b].add(a)
    cnt = 0
    for u in range(1, n + 1):
        ou = out[u]
        for v in ou:
            cnt += len(ou & out[v])
    return cnt

for (n, cor), want in examples:
    print(triangles_degree_oriented(n, cor), "kỳ vọng", want)

2 kỳ vọng 2
0 kỳ vọng 0


Code nộp LeetCode của Cách 4: xem mục 9.

**Độ phức tạp:** `O(E · √E)` thời gian trong trường hợp xấu nhất, Space `O(n + E)`. Với `E = 5·10⁴`: `√E ≈ 224` ⇒ ≈ `10⁷` thao tác trong giao tập (chạy trong C).

### 7.4 Cách 5 — Bitset (`int` Python)

`higher_bits[u]` có bit `v` bật nếu `v ∈ N(u)` và `v > u`. Số tam giác có cạnh nhỏ nhất `(u, v)` là `popcount(higher_bits[u] & higher_bits[v])`.

In [11]:
def triangles_bitset(n, corridors):
    hb = [0] * (n + 1)
    for a, b in corridors:
        if a > b:
            a, b = b, a
        hb[a] |= 1 << b                     # chỉ lưu láng giềng lớn hơn
    cnt = 0
    for a, b in corridors:
        if a > b:
            a, b = b, a
        cnt += (hb[a] & hb[b]).bit_count()
    return cnt

for (n, cor), want in examples:
    print(triangles_bitset(n, cor), "kỳ vọng", want)

2 kỳ vọng 2
0 kỳ vọng 0


**Độ phức tạp:** `O(E · n / 64)` phép AND từ máy (`n ≤ 1000` ⇒ 16 từ mỗi số) ≈ `8·10⁵` — rất nhanh. Space `O(n² / 64)` bit. Hợp với `n` nhỏ và đồ thị dày.

## 8. TRACE từng bước

Trace Cách 4 trên ví dụ 1: bậc, thứ hạng, định hướng cạnh, và tam giác được đếm ở đâu.

In [12]:
def trace_oriented(n, corridors):
    deg = [0] * (n + 1)
    for a, b in corridors:
        deg[a] += 1; deg[b] += 1
    print("bậc:", {v: deg[v] for v in range(1, n + 1)})
    order = sorted(range(1, n + 1), key=lambda v: (deg[v], v))
    print("thứ tự (thấp -> cao):", order)
    out = [set() for _ in range(n + 1)]
    for a, b in corridors:
        x, y = (a, b) if (deg[a], a) < (deg[b], b) else (b, a)
        out[x].add(y)
        print(f"  cạnh ({a},{b}) định hướng {x} -> {y}")
    cnt = 0
    for u in range(1, n + 1):
        for v in sorted(out[u]):
            common = out[u] & out[v]
            cnt += len(common)
            print(f"  xét {u}->{v}: out({u}) = {sorted(out[u])}, out({v}) = {sorted(out[v])}, chung = {sorted(common)}  | tổng = {cnt}")
    return cnt

trace_oriented(*examples[0][0])

bậc: {1: 3, 2: 3, 3: 2, 4: 3, 5: 1}
thứ tự (thấp -> cao): [5, 3, 1, 2, 4]
  cạnh (1,2) định hướng 1 -> 2
  cạnh (5,2) định hướng 5 -> 2
  cạnh (4,1) định hướng 1 -> 4
  cạnh (2,4) định hướng 2 -> 4
  cạnh (3,1) định hướng 3 -> 1
  cạnh (3,4) định hướng 3 -> 4
  xét 1->2: out(1) = [2, 4], out(2) = [4], chung = [4]  | tổng = 1
  xét 1->4: out(1) = [2, 4], out(4) = [], chung = []  | tổng = 1
  xét 2->4: out(2) = [4], out(4) = [], chung = []  | tổng = 1
  xét 3->1: out(3) = [1, 4], out(1) = [2, 4], chung = [4]  | tổng = 2
  xét 3->4: out(3) = [1, 4], out(4) = [], chung = []  | tổng = 2
  xét 5->2: out(5) = [2], out(2) = [4], chung = []  | tổng = 2


2

Bảng tóm tắt (ví dụ 1): bậc `deg = {1:3, 2:3, 3:2, 4:3, 5:1}` ⇒ thứ tự `5 < 3 < 1 < 2 < 4`.

| tam giác | đỉnh thấp nhất `x` | đỉnh giữa `y` | đỉnh cao nhất `z` | đếm khi xét |
|---|---|---|---|---|
| {1,3,4} | 3 | 1 | 4 | `3 → 1`, thấy `4 ∈ out(3) ∩ out(1)` |
| {1,2,4} | 1 | 2 | 4 | `1 → 2`, thấy `4 ∈ out(1) ∩ out(2)` |

Mỗi tam giác xuất hiện đúng một lần. ✓

In [13]:
def trace_ordered(n, corridors):
    adj = build_adj(n, corridors); cnt = 0
    for u in range(1, n + 1):
        for v in sorted(adj[u]):
            if v > u:
                ws = sorted(w for w in adj[u] & adj[v] if w > v)
                cnt += len(ws)
                if ws:
                    print(f"cạnh nhỏ nhất ({u},{v}) + đỉnh {ws} -> tam giác {[(u, v, w) for w in ws]}")
    print("tổng:", cnt)

trace_ordered(6, [[1, 2], [1, 3], [2, 3], [2, 4], [3, 4], [4, 5], [3, 5], [5, 6]])

cạnh nhỏ nhất (1,2) + đỉnh [3] -> tam giác [(1, 2, 3)]
cạnh nhỏ nhất (2,3) + đỉnh [4] -> tam giác [(2, 3, 4)]
cạnh nhỏ nhất (3,4) + đỉnh [5] -> tam giác [(3, 4, 5)]
tổng: 3


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def numberOfPaths(self, n: int, corridors: List[List[int]]) -> int:
        deg = [0] * (n + 1)
        for a, b in corridors:
            deg[a] += 1
            deg[b] += 1
        out = [set() for _ in range(n + 1)]
        for a, b in corridors:
            if (deg[a], a) < (deg[b], b):
                out[a].add(b)
            else:
                out[b].add(a)
        ans = 0
        for u in range(1, n + 1):
            ou = out[u]
            for v in ou:
                ans += len(ou & out[v])
        return ans

sol = Solution()
for (n, cor), want in examples:
    got = sol.numberOfPaths(n, cor)
    print(got, "OK" if got == want else "SAI")

2 OK
0 OK


## 10. Vì sao thuật toán đúng

**Cách 2.** Với tam giác `T = {a, b, c}`: `c ∈ N(a) ∩ N(b)`, `b ∈ N(a) ∩ N(c)`, `a ∈ N(b) ∩ N(c)` — nên `T` góp đúng `1` vào mỗi cạnh của nó, tổng `3`. Ngược lại, mỗi phần tử `w ∈ N(u) ∩ N(v)` với `(u, v)` là cạnh cho ra tam giác `{u, v, w}`. Vậy `Σ |N(u) ∩ N(v)| = 3 · (số tam giác)`. ✓

**Cách 3.** Ánh xạ "tam giác `{a < b < c}` ↦ bộ (`u = a`, `v = b`, `w = c`)" là song ánh giữa tập tam giác và tập bộ được đếm (`u < v < w`, ba cạnh đều có). ✓

**Cách 4.** Thứ hạng `(deg, chỉ số)` là **thứ tự toàn phần** (không có hai đỉnh bằng hạng). Mỗi tam giác có đỉnh hạng thấp nhất `x`, giữa `y`, cao nhất `z`. Định hướng: `x → y`, `x → z`, `y → z`. Khi xét cạnh có hướng `x → y`, `z ∈ out(x) ∩ out(y)` ⇒ đếm 1. Với cạnh `x → z`: cần đỉnh thứ ba trong `out(z)`, nhưng `y` có hạng thấp hơn `z` nên `y ∉ out(z)`. Với `y → z`: cần `x ∈ out(y)`, không đúng. ⇒ mỗi tam giác đếm đúng một lần. ✓

**Bổ đề chặn `|out(v)| ≤ √(2E)`.** Giả sử `|out(v)| = k`. Mọi đỉnh `w ∈ out(v)` có hạng cao hơn `v` nên `deg w ≥ deg v ≥ k`. Vậy có ít nhất `k` đỉnh bậc `≥ k` ⇒ tổng bậc `2E ≥ k²` ⇒ `k ≤ √(2E)`. Mỗi cạnh có hướng `u → v` tốn `O(min(|out u|, |out v|)) = O(√E)` ⇒ tổng `O(E √E)`. ✓

In [15]:
# Kiểm chứng bổ đề trên vài đồ thị ngẫu nhiên và đồ thị "ngôi sao" (một đỉnh bậc rất cao)
def max_out(n, corridors):
    deg = [0] * (n + 1)
    for a, b in corridors: deg[a] += 1; deg[b] += 1
    out = [0] * (n + 1)
    for a, b in corridors:
        out[a if (deg[a], a) < (deg[b], b) else b] += 1
    return max(out)

rng = random.Random(7)
star = [[1, v] for v in range(2, 1001)]
for name, (n, cor) in [("ngôi sao 1000 đỉnh", (1000, star)),
                       ("ngẫu nhiên E=5e4", (1000, [list(e) for e in rng.sample(list(combinations(range(1, 1001), 2)), 50000)]))]:
    E = len(cor)
    print(f"{name}: max |out(v)| = {max_out(n, cor)}  <=  sqrt(2E) = {(2 * E) ** 0.5:.0f}")

ngôi sao 1000 đỉnh: max |out(v)| = 1  <=  sqrt(2E) = 45
ngẫu nhiên E=5e4: max |out(v)| = 86  <=  sqrt(2E) = 316


## 11. Phân tích Time / Space

`n` = số phòng (`≤ 1000`), `E` = số hành lang (`≤ 5·10⁴`), `T` = số tam giác.

| Cách | Time | Space |
|---|---|---|
| 1. Bộ ba + ma trận | `O(n³)` (cắt tỉa `O(E n)`) | `O(n²)` |
| 2. Giao theo cạnh, chia 3 | `O(E √E)` | `O(n + E)` |
| 3. Có thứ tự `u < v < w` | `O(E √E + T)` | `O(n + E)` |
| 4. Định hướng theo bậc | `O(E √E)` | `O(n + E)` |
| 5. Bitset | `O(E n / 64)` | `O(n² / 64)` |

Đo trên ba đồ thị lớn: ngẫu nhiên (`n = 1000`, `E = 5·10⁴`), **đồ thị đầy đủ** `K_316` (`E = 49 770`, có `C(316, 3)` tam giác) và "bánh xe" (một đỉnh nối mọi đỉnh + chu trình).

In [16]:
rng = random.Random(1)
all_pairs = list(combinations(range(1, 1001), 2))
G_rand = (1000, [list(e) for e in rng.sample(all_pairs, 50000)])
G_full = (316, [[a, b] for a, b in combinations(range(1, 317), 2)])
wheel = [[1, v] for v in range(2, 1001)] + [[v, v + 1] for v in range(2, 1000)] + [[1000, 2]]
G_wheel = (1000, wheel)

for gname, G in [("random", G_rand), ("K_316", G_full), ("wheel", G_wheel)]:
    res = []
    for name, fn in [("edge/3", triangles_edge_div3), ("ordered", triangles_ordered),
                     ("degree", triangles_degree_oriented), ("bitset", triangles_bitset)]:
        if gname == "K_316" and name == "ordered":
            continue                                   # 5 triệu vòng lặp Python — bỏ để giữ thời gian chạy
        t0 = time.perf_counter(); r = fn(*G); res.append(r)
        print(f"{gname:7s} {name:8s}: {r:>8}  {time.perf_counter() - t0:.2f}s")
    assert len(set(res)) == 1
print("C(316,3) =", 316 * 315 * 314 // 6)

random  edge/3  :   166913  0.14s


random  ordered :   166913  0.13s
random  degree  :   166913  0.05s
random  bitset  :   166913  0.01s


K_316   edge/3  :  5209260  0.21s
K_316   degree  :  5209260  0.06s
K_316   bitset  :  5209260  0.01s
wheel   edge/3  :      999  0.00s
wheel   ordered :      999  0.00s
wheel   degree  :      999  0.00s
wheel   bitset  :      999  0.00s
C(316,3) = 5209260


## 12. Bộ test

In [17]:
fns = [triangles_edge_div3, triangles_ordered, triangles_degree_oriented, triangles_bitset,
       lambda n, c: Solution().numberOfPaths(n, c)]

for (n, cor), want in examples:
    assert triangles_brute(n, cor) == want
    for fn in fns:
        assert fn(n, cor) == want

edge = [
    ((3, [[1, 2], [2, 3], [3, 1]]), 1),                                  # một tam giác
    ((2, [[1, 2]]), 0),                                                  # n nhỏ nhất
    ((4, [[1, 2], [2, 3], [3, 4], [4, 1]]), 0),                          # chu trình 4 — không tính
    ((4, [[a, b] for a, b in combinations(range(1, 5), 2)]), 4),         # K4 có C(4,3) = 4
    ((5, [[a, b] for a, b in combinations(range(1, 6), 2)]), 10),        # K5
    ((5, [[1, 2], [1, 3], [1, 4], [1, 5]]), 0),                          # ngôi sao
    ((5, [[1, 2], [2, 3], [3, 1], [3, 4], [4, 5], [5, 3]]), 2),          # hai tam giác chung đỉnh 3
]
for (n, cor), want in edge:
    assert triangles_brute(n, cor) == want, (n, cor)
    for fn in fns:
        assert fn(n, cor) == want, (fn, n, cor)

rng = random.Random(2024)
for _ in range(400):
    n = rng.randint(2, 14)
    pairs = list(combinations(range(1, n + 1), 2))
    cor = [list(p) for p in rng.sample(pairs, rng.randint(1, len(pairs)))]
    for e in cor:
        if rng.random() < 0.5: e.reverse()                               # thứ tự hai đầu tùy ý
    rng.shuffle(cor)
    want = triangles_brute(n, cor)
    for fn in fns:
        assert fn(n, cor) == want, (fn, n, cor)
print("ví dụ + edge + 400 đồ thị ngẫu nhiên khớp brute force")

ví dụ + edge + 400 đồ thị ngẫu nhiên khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên chia 3 (hoặc chia sai hệ số)

In [18]:
def wrong_no_divide(n, corridors):
    adj = build_adj(n, corridors)
    return sum(len(adj[a] & adj[b]) for a, b in corridors)       # SAI: mỗi tam giác đếm 3 lần

def wrong_vertex_pairs_div3(n, corridors):
    adj = build_adj(n, corridors); s = 0
    for u in range(1, n + 1):
        for v in adj[u]:                                         # mỗi cạnh xét CẢ HAI chiều
            s += len(adj[u] & adj[v])
    return s // 3                                                # SAI: phải chia 6

for (n, cor), want in examples:
    print("không chia =", wrong_no_divide(n, cor), "| duyệt hai chiều chia 3 =", wrong_vertex_pairs_div3(n, cor), "| đúng =", want)

không chia = 6 | duyệt hai chiều chia 3 = 4 | đúng = 2
không chia = 0 | duyệt hai chiều chia 3 = 0 | đúng = 0


Duyệt `for u, for v in adj[u]` gặp mỗi cạnh **2 lần** ⇒ mỗi tam giác `3 × 2 = 6` lần. **Sửa:** chia đúng hệ số, hoặc tốt hơn là dùng ràng buộc thứ tự (Cách 3/4) để khỏi chia.

### Lỗi 2 — Chỉ thêm cạnh một chiều

In [19]:
def wrong_one_direction(n, corridors):
    adj = [set() for _ in range(n + 1)]
    for a, b in corridors:
        adj[a].add(b)                                            # SAI: thiếu adj[b].add(a)
    cnt = 0
    for u in range(1, n + 1):
        for v in adj[u]:
            if v > u:
                cnt += sum(1 for w in adj[u] & adj[v] if w > v)
    return cnt

for (n, cor), want in examples:
    print("sai =", wrong_one_direction(n, cor), "| đúng =", want)

sai = 0 | đúng = 2
sai = 0 | đúng = 0


Hành lang là **hai chiều**; input có thể cho `[5, 2]` với đỉnh lớn đứng trước. **Sửa:** thêm cả hai chiều (hoặc chuẩn hóa `a < b` trước khi lưu, như Cách 5).

### Lỗi 3 — Định hướng theo bậc nhưng **không phá hòa** bằng chỉ số

In [20]:
def wrong_tie(n, corridors):
    deg = [0] * (n + 1)
    for a, b in corridors: deg[a] += 1; deg[b] += 1
    out = [set() for _ in range(n + 1)]
    for a, b in corridors:
        if deg[a] <= deg[b]:                                     # SAI: bậc bằng nhau -> hướng phụ thuộc thứ tự input
            out[a].add(b)
        else:
            out[b].add(a)
    return sum(len(out[u] & out[v]) for u in range(1, n + 1) for v in out[u])

tri = (3, [[1, 2], [2, 3], [3, 1]])                              # ba đỉnh cùng bậc 2
print("tam giác 1-2-3: sai =", wrong_tie(*tri), "| đúng =", 1)
print("ví dụ 1:", wrong_tie(*examples[0][0]), "| đúng =", 2)

tam giác 1-2-3: sai = 0 | đúng = 1
ví dụ 1: 1 | đúng = 2


Với tam giác đều bậc, các cạnh có thể tạo thành **chu trình có hướng** `1 → 2 → 3 → 1` — không đỉnh nào "thấp nhất" ⇒ đếm `0`. **Sửa:** so sánh bộ `(deg, chỉ số)` — một thứ tự toàn phần nên không thể có chu trình có hướng.

### Lỗi 4 — Đếm cả "đường đi quay lại" thay vì tam giác (dùng `A³` không chia)

In [21]:
def wrong_trace_A3(n, corridors):
    M = [[0] * n for _ in range(n)]
    for a, b in corridors:
        M[a - 1][b - 1] = M[b - 1][a - 1] = 1
    A2 = [[sum(M[i][k] * M[k][j] for k in range(n)) for j in range(n)] for i in range(n)]
    return sum(A2[i][k] * M[k][i] for i in range(n) for k in range(n))    # trace(A^3) — SAI nếu không chia 6

for (n, cor), want in examples:
    t = wrong_trace_A3(n, cor)
    print("trace(A^3) =", t, "| trace/6 =", t // 6, "| đúng =", want)

trace(A^3) = 12 | trace/6 = 2 | đúng = 2
trace(A^3) = 0 | trace/6 = 0 | đúng = 0


`trace(A³)` đếm **chu trình có hướng có điểm xuất phát** độ dài 3 = `6 ×` số tam giác (đề coi chúng là một). **Sửa:** chia 6. (Cách ma trận còn tốn `O(n³)` — chỉ để minh họa.)

### Lỗi 5 — Dùng list làm danh sách kề rồi kiểm tra `in`

Đúng nhưng chậm: `w in adj_list[v]` là `O(deg v)` ⇒ có thể `O(E · n)`. **Sửa:** dùng `set` (hoặc bitset).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chu trình độ dài 3, khác nhau theo tập đỉnh" ⇒ **đếm tam giác**.
- Đếm theo cạnh ⇒ chia 3; đếm theo cặp (u, v có thứ tự) ⇒ chia 6; ép `u < v < w` ⇒ không chia.
- Đồ thị có thể có đỉnh bậc rất cao (ngôi sao) ⇒ định hướng theo bậc để chặn `O(E √E)`.
- `n ≤ ~5000` ⇒ bitset `int` Python là lựa chọn nhanh và gọn.
- Họ bài: 1761 *Minimum Degree of a Connected Trio in a Graph* (tam giác + bậc), 2242 *Maximum Score of a Node Sequence*, 2508 *Add Edges to Make Degrees of All Nodes Even*.

## 15. Biến thể của bài

1. **Liệt kê** các tam giác (không chỉ đếm): Cách 3/4 cho trực tiếp bộ ba.
2. **Đếm tam giác chứa mỗi đỉnh** (local triangle count — dùng cho hệ số phân cụm).
3. **1761 Minimum Degree of a Connected Trio:** với mỗi tam giác, tính `deg a + deg b + deg c - 6`, lấy nhỏ nhất.
4. **Đồ thị có hướng:** đếm chu trình có hướng `a → b → c → a`.
5. **Đếm chu trình độ dài 4** (hình vuông): với mỗi cặp đỉnh `(u, w)`, đếm số "đường 2 bước" `c(u, w)`, cộng `C(c, 2)`, chia 2.

In [22]:
def per_vertex_triangles(n, corridors):
    """Biến thể 2."""
    adj = build_adj(n, corridors); cnt = [0] * (n + 1)
    for u in range(1, n + 1):
        for v in adj[u]:
            if v > u:
                for w in adj[u] & adj[v]:
                    if w > v:
                        cnt[u] += 1; cnt[v] += 1; cnt[w] += 1
    return cnt[1:]

def min_trio_degree(n, corridors):
    """Biến thể 3 (LeetCode 1761)."""
    adj = build_adj(n, corridors); best = float("inf")
    for u in range(1, n + 1):
        for v in adj[u]:
            if v > u:
                for w in adj[u] & adj[v]:
                    if w > v:
                        best = min(best, len(adj[u]) + len(adj[v]) + len(adj[w]) - 6)
    return -1 if best == float("inf") else best

def count_4_cycles(n, corridors):
    """Biến thể 5."""
    adj = build_adj(n, corridors); total = 0
    for u in range(1, n + 1):
        c = {}
        for v in adj[u]:
            for w in adj[v]:
                if w > u:
                    c[w] = c.get(w, 0) + 1
        total += sum(k * (k - 1) // 2 for k in c.values())
    return total // 2

print("tam giác theo đỉnh (ví dụ 1):", per_vertex_triangles(*examples[0][0]))
print("min trio degree (ví dụ 1):", min_trio_degree(*examples[0][0]))
print("số chu trình 4 của K4:", count_4_cycles(4, [[a, b] for a, b in combinations(range(1, 5), 2)]), "(kỳ vọng 3)")

tam giác theo đỉnh (ví dụ 1): [2, 1, 1, 2, 0]
min trio degree (ví dụ 1): 2
số chu trình 4 của K4: 3 (kỳ vọng 3)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Bộ ba | định nghĩa | `O(n³)` | `O(n²)` | kiểm chứng, `n` nhỏ |
| 2. Giao theo cạnh / 3 | mỗi tam giác 3 cạnh | `O(E √E)` | `O(n + E)` | ngắn nhất |
| 3. Có thứ tự | `u < v < w` | `O(E √E + T)` | `O(n + E)` | cần liệt kê |
| 4. Định hướng bậc | đỉnh thấp nhất | `O(E √E)` | `O(n + E)` | **nộp bài**, đồ thị bất kỳ |
| 5. Bitset | popcount AND | `O(E n / 64)` | `O(n² / 64)` | `n` vừa, đồ thị dày |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `n = 6`, cạnh là mọi cặp trong `{1,2,3}` và mọi cặp trong `{4,5,6}`, thêm cạnh `(3, 4)`? Và đồ thị đầy đủ `K_6`?

**Bài 2 (tính tay):** với đồ thị `[[1,2],[1,3],[2,3],[2,4],[3,4]]` (`n = 4`), tính `|N(u) ∩ N(v)|` cho từng cạnh, tổng, và số tam giác.

**Bài 3 (điền TODO):** hoàn thiện `is_triangle(adj, a, b, c)` trả về `True` nếu ba phòng đôi một có hành lang.

**Bài 4 (tự viết lại):** viết `list_triangles(n, corridors)` trả về danh sách các bộ `(a, b, c)` với `a < b < c`, sắp xếp tăng dần.

**Bài 5 (mini challenge):** **đồ thị có hướng** — `corridors[i] = [a, b]` là hành lang **một chiều** `a → b`. Đếm số chu trình có hướng độ dài 3 (khác nhau theo tập phòng). Lưu ý `{a, b, c}` có thể chứa hai chu trình ngược chiều nếu có đủ cạnh hai chiều. Viết lời giải `O(E · √E)`-ish bằng giao tập và kiểm chứng bằng brute force. (Biến thể tự đặt — đối chiếu lại đề gốc nếu dùng làm đề thi.)

In [23]:
def is_triangle(adj, a, b, c):
    # TODO: trả về True nếu b ∈ adj[a], c ∈ adj[a] và c ∈ adj[b]
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Mỗi khối `{1,2,3}`, `{4,5,6}` là một tam giác. (2) Cạnh cầu `(3,4)` có `N(3) ∩ N(4) = ∅`. (3) `K_6`: chọn 3 trong 6 đỉnh.

**Bài 2:** (1) `N(1) = {2,3}`, `N(2) = {1,3,4}`, `N(3) = {1,2,4}`, `N(4) = {2,3}`. (2) Cạnh `(2,3)` có 2 láng giềng chung, các cạnh khác 1. (3) Tổng `6` ⇒ `2` tam giác.

**Bài 3:** (1) Ba phép `in` trên `set`. (2) Thứ tự `a, b, c` không quan trọng vì đồ thị vô hướng. (3) Dùng nó để viết lại brute force.

**Bài 4:** (1) Theo Cách 3. (2) Thu `(u, v, w)` thay vì đếm. (3) `sorted(...)` ở cuối.

**Bài 5:** (1) `out[a]`, `inn[a]`. (2) Chu trình `a → b → c → a` với `a` là đỉnh **nhỏ nhất** (để mỗi chu trình có hướng đếm một lần): với mỗi `b ∈ out[a]`, `b > a`, đếm `c ∈ out[b] ∩ inn[a]` với `c > a`. (3) Brute force: thử mọi `(a, b, c)` khác nhau với `a = min`, kiểm tra ba cạnh có hướng.

## 19. Đáp án tham khảo

In [24]:
# Bài 1
two_blocks = [[1, 2], [1, 3], [2, 3], [4, 5], [4, 6], [5, 6], [3, 4]]
assert Solution().numberOfPaths(6, two_blocks) == triangles_brute(6, two_blocks) == 2
K6 = [[a, b] for a, b in combinations(range(1, 7), 2)]
assert Solution().numberOfPaths(6, K6) == 20

# Bài 2
g = [[1, 2], [1, 3], [2, 3], [2, 4], [3, 4]]
A = build_adj(4, g)
per_edge = {(a, b): len(A[a] & A[b]) for a, b in g}
print("Bài 2:", per_edge, "tổng =", sum(per_edge.values()), "-> tam giác =", sum(per_edge.values()) // 3)
assert sum(per_edge.values()) == 6 and Solution().numberOfPaths(4, g) == 2

# Bài 3
def is_triangle_ans(adj, a, b, c):
    return b in adj[a] and c in adj[a] and c in adj[b]
assert is_triangle_ans(A, 1, 2, 3) and is_triangle_ans(A, 4, 3, 2) and not is_triangle_ans(A, 1, 2, 4)

# Bài 4
def list_triangles_ans(n, corridors):
    adj = build_adj(n, corridors); res = []
    for u in range(1, n + 1):
        for v in adj[u]:
            if v > u:
                for w in adj[u] & adj[v]:
                    if w > v:
                        res.append((u, v, w))
    return sorted(res)

print("Bài 4 (ví dụ 1):", list_triangles_ans(*examples[0][0]))
rng = random.Random(41)
for _ in range(200):
    n = rng.randint(3, 10); pairs = list(combinations(range(1, n + 1), 2))
    cor = [list(p) for p in rng.sample(pairs, rng.randint(1, len(pairs)))]
    adj = build_adj(n, cor)
    assert list_triangles_ans(n, cor) == [t for t in combinations(range(1, n + 1), 3) if is_triangle_ans(adj, *t)]
print("Bài 1-4: OK")

Bài 2: {(1, 2): 1, (1, 3): 1, (2, 3): 2, (2, 4): 1, (3, 4): 1} tổng = 6 -> tam giác = 2
Bài 4 (ví dụ 1): [(1, 2, 4), (1, 3, 4)]
Bài 1-4: OK


In [25]:
# Bài 5 — chu trình có hướng độ dài 3
def directed_triangles(n, corridors):
    out = [set() for _ in range(n + 1)]; inn = [set() for _ in range(n + 1)]
    for a, b in corridors:
        out[a].add(b); inn[b].add(a)
    cnt = 0
    for a in range(1, n + 1):
        for b in out[a]:
            if b > a:
                cnt += sum(1 for c in out[b] & inn[a] if c > a)    # a -> b -> c -> a, a nhỏ nhất
    return cnt

def directed_triangles_brute(n, corridors):
    E = {(a, b) for a, b in corridors}; cnt = 0
    for a in range(1, n + 1):
        for b in range(a + 1, n + 1):
            for c in range(a + 1, n + 1):
                if c != b and (a, b) in E and (b, c) in E and (c, a) in E:
                    cnt += 1
    return cnt

rng = random.Random(51)
for _ in range(300):
    n = rng.randint(2, 9)
    arcs = [(a, b) for a in range(1, n + 1) for b in range(1, n + 1) if a != b]
    cor = [list(x) for x in rng.sample(arcs, rng.randint(1, len(arcs)))]
    assert directed_triangles(n, cor) == directed_triangles_brute(n, cor)
print("Bài 5 ví dụ: 1->2->3->1 và 1->3->2->1:",
      directed_triangles(3, [[1, 2], [2, 3], [3, 1], [1, 3], [3, 2], [2, 1]]), "chu trình có hướng (cùng tập {1,2,3})")

Bài 5 ví dụ: 1->2->3->1 và 1->3->2->1: 2 chu trình có hướng (cùng tập {1,2,3})


Lưu ý ở Bài 5: nếu đề vẫn tính "khác nhau theo tập phòng" thì hai chu trình ngược chiều trên `{1,2,3}` gộp thành một — khi đó cần đếm **tập** `{a, b, c}` có ít nhất một chu trình có hướng. Đây là chỗ phải đọc kỹ đề biến thể.

## 20. Template nhớ nhanh + Checklist

```python
# Đếm tam giác O(E·sqrt(E)): định hướng cạnh từ đỉnh "thấp" sang "cao" theo (bậc, chỉ số)
deg = [0] * (n + 1)
for a, b in edges: deg[a] += 1; deg[b] += 1
out = [set() for _ in range(n + 1)]
for a, b in edges:
    if (deg[a], a) < (deg[b], b): out[a].add(b)
    else:                         out[b].add(a)
ans = sum(len(out[u] & out[v]) for u in range(1, n + 1) for v in out[u])

# Ngắn gọn (không định hướng): sum(len(adj[a] & adj[b]) for a, b in edges) // 3
```

**Checklist trước khi submit:**
- [ ] Đồ thị **vô hướng**: thêm cạnh hai chiều (hoặc chuẩn hóa `a < b`).
- [ ] Đúng hệ số: theo cạnh ⇒ `/3`; theo cặp có thứ tự ⇒ `/6`; ép thứ tự ⇒ không chia.
- [ ] Định hướng theo `(deg, chỉ số)` — phải phá hòa.
- [ ] Dùng `set`/bitset, không `in` trên list.
- [ ] Phòng đánh số từ `1` ⇒ mảng kích thước `n + 1`.